# Quick check for `data_preparation`
Small notebook to test SARIMAX-ready preparation and train/test split.


In [9]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'forecast_hourly_base').exists() and (ROOT.parent / 'forecast_hourly_base').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from forecast_hourly_base.data_preparation import (
    prepare_train_for_sarimax,
    prepare_inference_for_sarimax,
    split_panel_train_test,
)

#DATA_DIRS = [ROOT / 'src' / 'data', ROOT / 'data']
DATA_DIRS = [ROOT / 'src' / 'data', ROOT / 'data', ROOT / '..' / 'final_data', ROOT / 'final_data']
DATA_DIR = next((p for p in DATA_DIRS if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'No data directory found in: {DATA_DIRS}')

print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)

attraction_name = "Bumper Cars"

ROOT: /Users/alessandro_iva/Desktop/Eleven Hackathon/src
DATA_DIR: /Users/alessandro_iva/Desktop/Eleven Hackathon/src/../final_data


In [10]:
pack = prepare_train_for_sarimax(data_dir=DATA_DIR, attraction_name = attraction_name, train_ratio=0.8)
full_df = pack['full_df']
train_df = pack['train_df']
test_df = pack['test_df']
exog_cols = pack['exog_cols']

print('full shape:', full_df.shape)
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('split timestamp:', pack['split_timestamp'])
print('num exog cols:', len(exog_cols))
print('num dummy cols:', len(pack['dummy_cols_created']))
full_df.head()


full shape: (13469, 44)
train shape: (10775, 44)
test shape: (2694, 44)
split timestamp: 2021-12-28 20:00:00
num exog cols: 41
num dummy cols: 31


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,covid_1
0,2018-07-21 09:00:00,Bumper Cars,5.0,59066.0,20.59,1014,68,3.43,62,0.0,...,0,0,0,1,0,0,0,0,0,0
1,2018-07-21 10:00:00,Bumper Cars,5.0,59066.0,22.28,1014,62,2.41,66,0.0,...,0,0,0,1,0,0,0,0,0,0
2,2018-07-21 11:00:00,Bumper Cars,12.5,59066.0,23.20,1014,58,2.39,59,0.0,...,0,0,0,1,0,0,0,0,0,0
3,2018-07-21 12:00:00,Bumper Cars,17.5,59066.0,24.70,1014,56,2.10,34,0.0,...,0,0,0,1,0,0,0,0,0,0
4,2018-07-21 13:00:00,Bumper Cars,12.5,59066.0,24.71,1014,54,2.27,51,0.0,...,0,0,0,1,0,0,0,0,0,0


## Testing

In [11]:
test_df.head()

,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,covid_1
0,2021-12-28 21:00:00,Bumper Cars,5.00,49787.0,10.48,1007,82,6.40,91,0.00,...,0,0,0,0,0,0,0,0,1,1
1,2021-12-29 09:00:00,Bumper Cars,5.00,45147.0,9.48,1009,99,4.06,100,1.01,...,0,0,0,0,0,0,0,0,1,1
2,2021-12-29 10:00:00,Bumper Cars,11.25,45147.0,10.61,1009,99,3.83,100,0.26,...,0,0,0,0,0,0,0,0,1,1
3,2021-12-29 11:00:00,Bumper Cars,27.50,45147.0,11.73,1009,99,6.23,100,0.00,...,0,0,0,0,0,0,0,0,1,1
4,2021-12-29 12:00:00,Bumper Cars,37.50,45147.0,13.19,1010,99,6.38,100,0.15,...,0,0,0,0,0,0,0,0,1,1


In [12]:
print('train max time:', train_df['date_hour'].max())
print('test  min time:', test_df['date_hour'].min())
print('time split OK:', train_df['date_hour'].max() <= test_df['date_hour'].min())

train max time: 2021-12-28 20:00:00
test  min time: 2021-12-28 21:00:00
time split OK: True


## Inference

In [13]:
weather_forecast_df = (
    pd.read_csv(
        DATA_DIR / 'weather_data.csv',
        usecols=['dt_iso', 'temp', 'pressure', 'humidity', 'wind_speed', 'clouds_all', 'rain_1h', 'visibility'],
    )
    .tail(24 * 14)
    .copy()
)

attendance_forecast_df = (
    pd.read_csv(DATA_DIR / 'attendance.csv')
    .query("FACILITY_NAME == 'PortAventura World'")
    .assign(date=lambda d: pd.to_datetime(d['USAGE_DATE'], errors='coerce').dt.floor('D'))
    [['date', 'attendance']]
    .dropna()
    .drop_duplicates('date')
    .tail(7)
)

previous_week_real_df = (
    full_df[full_df['ENTITY_DESCRIPTION_SHORT'].astype(str) == attraction_name]
    [['date_hour', 'guests_sum', 'availability', 'utilization']]
    .tail(24 * 14)
    .copy()
)

print('attraction:', attraction_name)
print('weather rows:', len(weather_forecast_df))
print('attendance days:', len(attendance_forecast_df))
print('previous-week rows:', len(previous_week_real_df))


attraction: Bumper Cars
weather rows: 336
attendance days: 7
previous-week rows: 336


In [14]:
# Load daily forecasts from XGBoost model
FORECAST_DIRS = [ROOT / 'final_data' / 'forecasts', ROOT / '..' / 'final_data' / 'forecasts']
FORECAST_DIR = next((p for p in FORECAST_DIRS if p.exists()), None)
if FORECAST_DIR is None:
    raise FileNotFoundError(f'No forecasts directory found in: {FORECAST_DIRS}')

xgb_daily = pd.read_csv(FORECAST_DIR / 'daily' / 'xgboost_daily_oneweek.csv')
xgb_daily['date'] = pd.to_datetime(xgb_daily['date'])

attendance_forecast_df = xgb_daily[['date', 'predicted']].rename(columns={'predicted': 'attendance'})
print(f'attendance_forecast_df: {len(attendance_forecast_df)} daily rows from XGBoost daily model')
attendance_forecast_df

attendance_forecast_df: 29 daily rows from XGBoost daily model


,date,attendance
0,2022-07-26,45168.464844
1,2022-07-27,44844.699219
2,2022-07-28,44950.613281
3,2022-07-29,44329.453125
4,2022-07-30,52459.949219
5,2022-07-31,45359.386719
6,2022-08-01,44423.214844
7,2022-08-02,43053.878906
8,2022-08-03,42342.273438
9,2022-08-04,41658.625000


In [15]:
inference_df = prepare_inference_for_sarimax(
    weather_forecast_df=weather_forecast_df,
    attendance_forecast_df=attendance_forecast_df,
    attraction_name=attraction_name,
    previous_week_real_df=previous_week_real_df,
    train_feature_cols=exog_cols,
)

missing_exog = [c for c in exog_cols if c not in inference_df.columns]
print('inference shape:', inference_df.shape)
print('missing exog cols:', len(missing_exog))
inference_df.head()


inference shape: (196, 44)
missing exog cols: 0


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,covid_1
0,2022-08-10 09:00:00,Bumper Cars,NaN,43252.152344,26.47,1022,38,3.84,0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2022-08-10 10:00:00,Bumper Cars,NaN,43252.152344,29.43,1022,37,3.94,0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2022-08-10 11:00:00,Bumper Cars,NaN,43252.152344,30.94,1021,31,4.88,4,0.0,...,0,0,0,0,0,0,0,0,0,0
3,2022-08-10 12:00:00,Bumper Cars,NaN,43252.152344,31.47,1021,30,5.14,31,0.0,...,0,0,0,0,0,0,0,0,0,0
4,2022-08-10 13:00:00,Bumper Cars,NaN,43252.152344,31.82,1020,30,5.08,35,0.0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# Build LightGBM-based attendance forecast (weekly → daily)
lgbm_weekly = pd.read_csv(FORECAST_DIR / 'weekly' / 'lightgbm_quantile_regression.csv')
lgbm_weekly['week'] = pd.to_datetime(lgbm_weekly['week'])

daily_rows = []
for _, row in lgbm_weekly.iterrows():
    week_start = row['week']
    daily_att = row['predicted'] / 7
    for d in range(7):
        daily_rows.append({
            'date': week_start + pd.Timedelta(days=d),
            'attendance': daily_att,
        })

attendance_forecast_df_lgbm = pd.DataFrame(daily_rows)
print(f'LightGBM attendance_forecast_df: {len(attendance_forecast_df_lgbm)} daily rows')

# Create inference_df copy with LightGBM attendance
inference_df_lgbm = prepare_inference_for_sarimax(
    weather_forecast_df=weather_forecast_df,
    attendance_forecast_df=attendance_forecast_df_lgbm,
    attraction_name=attraction_name,
    previous_week_real_df=previous_week_real_df,
    train_feature_cols=exog_cols,
)

print(f'inference_df_lgbm shape: {inference_df_lgbm.shape}')
print(f'\nXGBoost attendance sample:  {inference_df["attendance"].iloc[:3].values}')
print(f'LightGBM attendance sample: {inference_df_lgbm["attendance"].iloc[:3].values}')
inference_df_lgbm.head()

LightGBM attendance_forecast_df: 371 daily rows
inference_df_lgbm shape: (196, 44)

XGBoost attendance sample:  [43252.15234375 43252.15234375 43252.15234375]
LightGBM attendance sample: [43209.28571429 43209.28571429 43209.28571429]


,date_hour,ENTITY_DESCRIPTION_SHORT,wait_time_avg,attendance,temp,pressure,humidity,wind_speed,clouds_all,rain_1h,...,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12,covid_1
0,2022-08-10 09:00:00,Bumper Cars,NaN,43209.285714,26.47,1022,38,3.84,0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2022-08-10 10:00:00,Bumper Cars,NaN,43209.285714,29.43,1022,37,3.94,0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,2022-08-10 11:00:00,Bumper Cars,NaN,43209.285714,30.94,1021,31,4.88,4,0.0,...,0,0,0,0,0,0,0,0,0,0
3,2022-08-10 12:00:00,Bumper Cars,NaN,43209.285714,31.47,1021,30,5.14,31,0.0,...,0,0,0,0,0,0,0,0,0,0
4,2022-08-10 13:00:00,Bumper Cars,NaN,43209.285714,31.82,1020,30,5.08,35,0.0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# Save both inference DataFrames to final_data/merged/
MERGED_DIRS = [ROOT / 'final_data' / 'merged', ROOT / '..' / 'final_data' / 'merged']
MERGED_DIR = next((p for p in MERGED_DIRS if p.exists()), None)
if MERGED_DIR is None:
    raise FileNotFoundError(f'No merged directory found in: {MERGED_DIRS}')

inference_df.to_csv(MERGED_DIR / 'inference_xgboost.csv', index=False)
inference_df_lgbm.to_csv(MERGED_DIR / 'inference_lgbm.csv', index=False)

print(f'Saved inference_xgboost.csv ({len(inference_df)} rows) to {MERGED_DIR}')
print(f'Saved inference_lgbm.csv ({len(inference_df_lgbm)} rows) to {MERGED_DIR}')

Saved inference_xgboost.csv (196 rows) to /Users/alessandro_iva/Desktop/Eleven Hackathon/src/../final_data/merged
Saved inference_lgbm.csv (196 rows) to /Users/alessandro_iva/Desktop/Eleven Hackathon/src/../final_data/merged


In [16]:
train2, test2 = split_panel_train_test(full_df, train_ratio=0.8, time_col='date_hour')
print('split_panel_train_test shapes:', train2.shape, test2.shape)
print('same split as wrapper:', train2.shape == train_df.shape and test2.shape == test_df.shape)

split_panel_train_test shapes: (10775, 44) (2694, 44)
same split as wrapper: True
